# 03 · Feature Engineering

Produces `dgstats_panel.parquet`, `caiso_daily.parquet`, and `caiso_monthly.parquet`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'

import pandas as pd
from cleaning import aggregate_dgstats_panel
from feature_engineering import compute_net_load, compute_curtailment, aggregate_caiso_daily, aggregate_caiso_monthly

## 3.1 DGStats ZIP × Year Panel

In [ ]:
dgstats = pd.read_parquet(PROC / 'dgstats_clean.parquet')
panel = aggregate_dgstats_panel(dgstats)
print(f'DGStats panel: {len(panel):,} ZIP-year rows | {panel["zip_code"].nunique():,} unique ZIPs')
panel.head()

In [ ]:
panel.to_parquet(PROC / 'dgstats_panel.parquet', index=False)

## 3.2 CAISO Net Load & Curtailment

In [ ]:
caiso = pd.read_parquet(PROC / 'caiso_clean.parquet')

# Standardize column names from gridstatus output
# gridstatus returns columns like: 'Time', 'Solar', 'Wind', 'Load', etc.
# Map to our internal naming convention
col_map = {
    'Time': 'interval_start_utc',
    'Solar': 'solar_mw',
    'Wind': 'wind_mw',
    'Load': 'demand_mw',
    'Natural Gas': 'gas_mw',
    'Imports': 'imports_mw',
}
caiso = caiso.rename(columns={k: v for k, v in col_map.items() if k in caiso.columns})

# Ensure required columns exist with fallbacks
for col in ['solar_mw', 'wind_mw', 'demand_mw']:
    if col not in caiso.columns:
        print(f'WARNING: {col} not found — columns available: {list(caiso.columns)}')

caiso.head(3)

In [ ]:
daily = aggregate_caiso_daily(caiso)
print(f'CAISO daily: {len(daily):,} rows')
daily.head()

In [ ]:
monthly = aggregate_caiso_monthly(daily)
print(f'CAISO monthly: {len(monthly):,} rows')
monthly.head()

In [ ]:
daily.to_parquet(PROC / 'caiso_daily.parquet', index=False)
monthly.to_parquet(PROC / 'caiso_monthly.parquet', index=False)
print('Saved caiso_daily.parquet and caiso_monthly.parquet')

## 3.3 DeepSolar Tract → ZIP Aggregation

In [ ]:
from pathlib import Path
RAW = ROOT / 'data' / 'raw'

deepsolar = pd.read_parquet(RAW / 'deepsolar_ca_tracts.parquet')
xwalk = pd.read_parquet(RAW / 'zip_tract_xwalk.parquet')

# Standardize FIPS to 11-char string
deepsolar['fips'] = deepsolar['fips'].astype(str).str.zfill(11)
xwalk['tract'] = xwalk['tract'].astype(str).str.zfill(11)

# Merge on tract FIPS
ds_zip = deepsolar.merge(xwalk, left_on='fips', right_on='tract', how='inner')

# Area-weighted aggregation (weight by res_ratio if available)
weight_col = 'res_ratio' if 'res_ratio' in ds_zip.columns else None
numeric_cols = [c for c in ['solar_system_count', 'solar_panel_area_divided_by_area'] if c in ds_zip.columns]

ds_agg = ds_zip.groupby('zip')[numeric_cols].sum().reset_index().rename(columns={'zip': 'zip_code'})
ds_agg['zip_code'] = ds_agg['zip_code'].astype(str).str.zfill(5)
print(f'DeepSolar → ZIP: {len(ds_agg):,} ZIPs')
ds_agg.to_parquet(PROC / 'deepsolar_zip.parquet', index=False)